# 駅周辺の範囲定義：656駅で確かめる

Notebook 11の吉祥寺の結果は変更せず、他の駅での分布を調べます。
「距離上限の変更」と「同じ距離でも定義が違うこと」を分けます。

固定OSM道路・保存済みOvertureだけを使用します。初回の全駅計算は[実行手順](../docs/walking-sensitivity-validation.md)を参照してください。
[事前設計](../docs/walking-sensitivity-design.md) / [記事資料](../docs/walking-sensitivity-article-handoff.md)

## 1. 検証済み結果を読む
駅代表点・POI双方の道路接続上限100m。道路距離には接続距離を含めます。
7条件は1km四方、直線400/500/800m、道路400/500/800mです。

In [ ]:
from geoai_open_lab.walking_sensitivity import load_results, OUT
from IPython.display import display, SVG
audit, conditions, pairs, distributions = load_results()
print(f"対象 {len(audit)}駅 / 道路比較有効 {audit.valid.sum()}駅")

## 2. 距離上限と定義方法を分けて比較
Jaccard = 両条件に共通するPOI数 ÷ どちらかに入るPOI数。
1なら同じ集合です。和集合30件未満は、少数の違いが値を大きく動かすため別集計します。閾値は結果を見る前に固定しました。

In [ ]:
display(distributions[distributions.cohort.eq('all')][['first','second','valid_stations','jaccard_median','jaccard_p10','jaccard_p90']])
display(distributions[distributions.cohort.eq('union_lt30')][['first','second','valid_stations','jaccard_median']])

## 3. 吉祥寺は全体のどこにあるか
正方形と道路800mは距離も定義も違います。Notebook 11との連続性のための比較であり、定義だけの効果とは呼びません。
同じ500mで比較する行も確認します。

In [ ]:
display(pairs[pairs.region_id.eq('n02-003584')][['first','second','n_first','n_second','shared','jaccard','composition_tvd']])

## 4. 分布・対照駅・規則で選んだ例
図は全駅の集計から生成します。太線はP10–P90、赤菱形は吉祥寺。
TVDはカテゴリ構成比の差をまとめた値（0=同じ、1=完全に異なる）です。

In [ ]:
from geoai_open_lab.walking_sensitivity_figures import figures
figures()
for name in ['distributions','count_composition','controls','examples']:
    display(SVG(filename=str(OUT / (name + '.svg'))))

## 5. 言えること・言えないこと
- 吉祥寺1駅の高一致を全駅に一般化せず、分布と少数POI駅を確認します。
- 登録数は営業中店舗数ではありません。OSMとPOIには欠落・時点差があります。
- 駅代表点は実際の出口ではなく、接続線も実際の通路を保証しません。
- 道路到達圏を正解とせず、線路・河川等を低一致の原因と断定しません。
- 今回は**分析範囲の定義に対する感度**です。固定外枠の再分割ではなく、古典的MAUPのゾーニング効果を分離していません。
- UGCoPの地理的文脈の選択に関係しますが、人の活動範囲・時間・因果効果を測る実験ではありません。高一致もMAUPの反証ではありません。
- 近接駅のAOIは重複するため656の独立標本ではありません。

© OpenStreetMap contributors / ODbL 1.0、Overture 2026-09-23.0、駅N02-25。
公開Notebookは出力なし。個別POI・道路・生成CSV・図・実行済みHTMLはGit管理外です。